# Module 5 – Use Case 1: Proposal Assistant (Workspace)

---

## 🔧 Setup and Configuration

Before we begin, let's import the shared configuration and set up a simple helper function for testing LLM calls.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from pathlib import Path
from typing import Dict, List, Any


load_dotenv(dotenv_path=Path(r"C:\Users\shonr\OneDrive - Tekframeworks\Secret_keys\.env"), override=False)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_LLM_DEPLOYMENT = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

def validate_config():
    missing=[]
    for k,v in {
        "AZURE_OPENAI_ENDPOINT":AZURE_OPENAI_ENDPOINT,
        "AZURE_OPENAI_API_KEY":AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_LLM_DEPLOYMENT":AZURE_OPENAI_LLM_DEPLOYMENT,
        "AZURE_OPENAI_API_VERSION":AZURE_OPENAI_API_VERSION
    }.items():
        if not v:
            missing.append(k)
    if missing:
        raise ValueError(f"Missing environment variables: {missing}")
    print("CONFIG LOADED SUCCESSFULLY")


# ============================================================
# VALIDATE CONFIG
# ============================================================
def validate_config():
    missing = []

    if not AZURE_OPENAI_ENDPOINT:
        missing.append("AZURE_OPENAI_ENDPOINT")
    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")
    if not AZURE_OPENAI_LLM_DEPLOYMENT:
        missing.append("AZURE_OPENAI_LLM_DEPLOYMENT")
    if not AZURE_OPENAI_API_VERSION:
        missing.append("API_VERSION")

    if missing:
        raise ValueError(f"Missing environment variables: {', '.join(missing)}")


# ============================================================
# EXECUTION CHECK
# ============================================================
try:
    validate_config()

    print("✅ Configuration loaded successfully!")
    print(f"   Using model: {AZURE_OPENAI_LLM_DEPLOYMENT}")

except Exception as e:
    print(f"⚠️ Configuration error: {e}")
    print("   Ensure your .env file exists and contains required variables.")

### Optional Helper Function

If you want to test LLM calls directly while designing your solution, you can use this simple helper function:

In [ ]:
def call_llm(prompt: str, temperature: float = 0.7, max_tokens: int = 500) -> str:
    """
    Simple helper to make a single LLM call.
    
    Args:
        prompt: The prompt text to send to the model
        temperature: Controls randomness (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum length of the response
    
    Returns:
        The model's response as a string
    """
    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_LLM_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
        "x-ms-model-mesh-model-name": AZURE_OPENAI_LLM_DEPLOYMENT,
    }
    
    body = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    data = response.json()
    
    return data["choices"][0]["message"]["content"]


# Quick test (optional - uncomment to run)
test_response = call_llm("Say hello in one sentence.")
print(f"Test response: {test_response}")

---

## 🧠 Quick Recap & How to Use This Notebook

### Quick Recap of Patterns

Before diving into the use case, let's quickly review the key LLM usage patterns from `M5_Intro_Agents_and_Pattern.ipynb`:

| Pattern | What it is | When to use |
|---------|-----------|-------------|
| **LLM call** | Single request-response | One-off transformations, simple Q&A |
| **Chat + memory** | Multi-turn conversation with context | Interactive conversations, iterative refinement |
| **Prompt app** | Parameterized prompt template | Reusable, consistent transformations |
| **Workflow / chain** | Fixed sequence of LLM calls | Multi-step pipelines with predictable flow |
| **Agent** | LLM that chooses tools/actions dynamically | Dynamic problem-solving, adaptive behavior |
| **Agentic system** | Multiple agents with feedback loops | Complex scenarios requiring iteration |
| **RAG** | Retrieve context before responding | Grounding responses in external knowledge |

**💡 Need a deeper refresher?** Open `M5_Intro_Agents_and_Patterns.ipynb` to review examples and code for each pattern.

### How to Use This Workspace

Follow these steps to get the most out of this notebook:

1. **📖 Read the use case** (Section 3): Understand the business problem and requirements
2. **🧩 Choose your architecture** (Section 4): Decide between prompt app, workflow, agent, or hybrid approach
3. **✏️ Design your solution** (Section 5): Define inputs/outputs, steps/roles, and evaluation criteria
4. **💻 (Optional) Start coding** (Section 6): Sketch a partial implementation if you want hands-on practice
5. **🔍 Prepare for comparison** (Section 7): Get ready to see the reference solution
6. **💭 Reflect** (Section 8): Capture your learnings and questions

**Remember:** This is a thinking and design space. You don't need to build a complete solution here—that's what the reference solution notebook is for!

---

## 📘 Use Case Description & Requirements

### The Business Problem

You are building an **AI-powered Proposal Assistant** that generates client-ready proposals for GenAI and Azure-based solutions.

**Context:**
- Your organization receives requests from internal business units or external clients who want to implement GenAI solutions
- These requests come in various forms: RFPs, informal emails, requirement documents, or meeting notes
- Your team needs to quickly produce professional, comprehensive proposals that address these requirements

**The Challenge:**
- Manual proposal writing is time-consuming (4-8 hours per proposal)
- Quality and consistency vary depending on who writes it
- Proposals need to balance technical depth with business value
- Must include standard sections while being customized to each client's needs

### Expected Output Structure

A complete proposal typically includes the following sections:

1. **Executive Summary / Introduction**
   - High-level overview of the proposed solution
   - Key benefits and value proposition

2. **Business Context & Problem Statement**
   - Understanding of the client's domain and challenges
   - Current state and pain points

3. **Proposed GenAI Solution Overview**
   - Solution architecture at a conceptual level
   - Key components and how they work together
   - Technology stack (Azure services, LLM models, etc.)

4. **Implementation Approach**
   - Phased delivery plan
   - Milestones and deliverables
   - Timeline estimates

5. **Effort Estimation & Resource Requirements**
   - High-level effort breakdown
   - Team composition and roles
   - Rough cost estimates (optional)

6. **Risks, Assumptions & Mitigation Strategies**
   - Technical risks and dependencies
   - Business assumptions
   - Proposed mitigation approaches

7. **Next Steps & Call to Action**
   - Immediate next steps
   - Decision points
   - Contact information

### Requirement Intake Template

When gathering requirements from stakeholders, capture the following information:

**Client / Business Unit:**
- _[Name of the client organization or internal business unit]_

**Domain / Industry:**
- _[e.g., Healthcare, Finance, Retail, Manufacturing]_

**Problem Statement:**
- _[What business problem are they trying to solve?]_
- _[What are the key pain points?]_

**Desired Outcomes:**
- _[What does success look like?]_
- _[What metrics or KPIs matter?]_

**Constraints:**
- **Budget:** _[Any budget limitations?]_
- **Technology:** _[Existing tech stack, cloud provider preferences]_
- **Compliance:** _[Regulatory requirements, data privacy needs]_
- **Timeline:** _[When do they need this?]_

**Non-Functional Requirements:**
- **Performance:** _[Latency, throughput expectations]_
- **Security:** _[Data protection, access control]_
- **Scalability:** _[Expected growth, concurrent users]_
- **Integration:** _[Systems to integrate with]_

### Sample Requirement (Starting Point)

Below is a sample requirement you can use as a starting point. Feel free to modify it or replace it with your own scenario!

---

**📋 Sample Requirement: Document Intelligence Assistant**

**Client / Business Unit:** Legal Operations Department, ABC Financial Services

**Domain / Industry:** Financial Services / Banking

**Problem Statement:**
Our legal team processes hundreds of contracts, compliance documents, and regulatory filings each month. Currently, lawyers spend 40-60% of their time on manual document review: extracting key clauses, identifying risks, checking compliance, and summarizing findings for stakeholders. This is unsustainable as document volume grows 20% year-over-year.

**Desired Outcomes:**
- Reduce document review time by 50-70%
- Improve consistency in risk identification across reviewers
- Enable faster response to regulatory queries
- Free up lawyers to focus on high-value strategic work
- Maintain or improve accuracy (currently 95% accuracy on manual review)

**Constraints:**
- **Budget:** $200K-$300K for MVP (6-month project)
- **Technology:** Must use Azure (existing enterprise agreement), prefer Azure OpenAI Service
- **Compliance:** Subject to financial services regulations (SOC 2, data residency requirements)
- **Timeline:** Need MVP in production within 6 months

**Non-Functional Requirements:**
- **Performance:** Process a 50-page contract in under 2 minutes
- **Security:** End-to-end encryption, role-based access control, audit logging
- **Scalability:** Handle 500 documents per day initially, scale to 2000/day within 12 months
- **Integration:** Must integrate with existing document management system (SharePoint) and case management tool

---

**💡 Your turn:** You can use this sample requirement as-is, modify it for a different domain, or create your own scenario based on a real need from your organization.

---

## 🧩 Architecture Thinking: Choose Your Approach

Now that you understand the requirements, it's time to think about **how** to build this solution. Let's review your architecture options:

### Pattern Options Quick Reference

**1. Prompt App** (1-2 LLM calls, parameterized prompt)
- Takes requirement text as input
- Uses a single well-crafted prompt template
- Generates complete proposal in one shot
- **Pros:** Simple, fast, low cost
- **Cons:** Limited flexibility, hard to control structure, no iteration

**2. Workflow / Chain** (Fixed multi-step pipeline)
- Breaks proposal generation into discrete steps (e.g., outline → draft sections → format)
- Each step is predetermined and runs in sequence
- **Pros:** More control, easier to debug, consistent structure
- **Cons:** No adaptation based on quality, more LLM calls

**3. Agent** (Dynamic decision-making with tools)
- Uses LLM to decide which actions to take (plan, draft, refine, etc.)
- Can access tools (e.g., retrieve templates, validate content)
- Makes runtime decisions based on context
- **Pros:** Flexible, can handle varied requirements
- **Cons:** Unpredictable paths, harder to debug, potentially more calls

**4. Hybrid** (Mix of deterministic and agentic)
- Fixed workflow for well-understood steps
- Agent/adaptive behavior for specific sub-tasks (e.g., quality evaluation, section refinement)
- **Pros:** Balances control with flexibility
- **Cons:** More complex to design and implement

### Decision Criteria

Use this table to think through which approach makes sense for your scenario:

| Criterion                  | Rating (Low/Med/High) | Your Notes |
|---------------------------|-----------------------|------------|
| **Cost sensitivity**<br>_Budget constraints, need to minimize LLM calls_ | | |
| **Correctness criticality**<br>_How critical is it that every proposal is perfect?_ | | |
| **Need for iteration/refinement**<br>_Do proposals need multiple revisions?_ | | |
| **Variability in requirements**<br>_How different are requirements from each other?_ | | |
| **Risk / compliance constraints**<br>_Need for audit trails, predictable behavior_ | | |
| **Latency requirements**<br>_How fast does the proposal need to be ready?_ | | |
| **Maintainability**<br>_Ease of debugging, testing, and updating_ | | |

### Your Architecture Choice

Based on your analysis above, document your initial decision:

**I will start with:** _[Prompt app / Workflow / Agent / Hybrid]_

**Reasoning (3-5 sentences):**
- _[Why does this approach fit the requirements best?]_
- _[What are the key trade-offs you're accepting?]_
- _[Are there specific parts of the problem that influenced your choice?]_

---

**💡 Note:** You will later see reference implementations of **both** a workflow-style and an agentic-style solution. This will help you understand the practical differences between these approaches.

---

## ✏️ Design Your Solution

Now let's get specific about **how** your solution will work. This section helps you think through inputs, outputs, steps/roles, and evaluation before writing code.

### 5.1 Inputs & Outputs

Define exactly what your system accepts as input and what it produces as output.

#### Input Design

**Format:**
- _[Free text paragraph? Structured JSON? Form fields? A mix?]_

**Required fields:**
- _[List the minimum information needed to generate a proposal]_

**Optional fields:**
- _[What additional information would improve the proposal if available?]_

**Example input format:**
```
[Show what a typical input would look like]
```

#### Output Design

**Structure (sections / headings):**
- _[Will you use the 7-section template? Fewer sections? Custom structure?]_

**Level of detail:**
- _[Executive summary: 1 paragraph or 2 pages?]_
- _[Technical sections: high-level overview or detailed specs?]_

**Formatting / style constraints:**
- _[Professional business language? Include tables/charts? Markdown format?]_

**Example output format:**
```
[Show what a typical output structure would look like]
```

### 5.2 Steps / Roles / Tools

Depending on your chosen architecture (workflow vs agent), you'll think about this differently. **Choose the template that matches your approach** or fill out both if you're exploring multiple options.

#### Option A: Workflow-Style Design

If you chose a **workflow/chain** approach, break down the proposal generation into discrete, sequential steps:

| Step | Description | LLM Call? | Inputs | Outputs | Tools/Data Used |
|------|-------------|-----------|--------|---------|-----------------|
| 1 | _e.g., Generate outline_ | Yes | Requirement text | Structured outline | Proposal template |
| 2 | _e.g., Draft executive summary_ | Yes | Requirement + outline | Summary text | None |
| 3 | _e.g., Draft problem statement_ | Yes | Requirement + outline | Problem section | None |
| 4 | _e.g., Draft solution overview_ | Yes | Requirement + outline | Solution section | Azure architecture patterns |
| 5 | _e.g., Draft implementation plan_ | Yes | Requirement + outline | Implementation section | None |
| 6 | _e.g., Draft risks & assumptions_ | Yes | Requirement + outline | Risks section | None |
| 7 | _e.g., Combine & format_ | No/Yes | All sections | Final proposal document | Formatting template |

**Estimated LLM calls:** _[Fill in your estimate]_

**Total estimated tokens:** _[Rough guess if you know input/output sizes]_

#### Option B: Agent-Style Design

If you chose an **agent** or **agentic** approach, define the roles/agents and their responsibilities:

| Agent / Role | Responsibility | Inputs | Outputs | Tools Available | Decision Logic |
|--------------|---------------|--------|---------|-----------------|----------------|
| **Planner** | _Analyze requirement and create proposal plan_ | Requirement text | Structured plan/outline | Proposal templates, best practices | _Determines sections needed based on requirement complexity_ |
| **Drafter** | _Generate content for each section_ | Requirement + plan | Draft sections | Domain knowledge, examples | _Chooses writing style based on audience_ |
| **Evaluator** | _Score proposal quality and provide feedback_ | Requirement + draft | Quality score + feedback | Rubric, checklist | _Decides if revision needed_ |
| **Refiner** _(optional)_ | _Improve draft based on feedback_ | Draft + feedback | Refined draft | Style guide | _Applies specific improvements_ |

**Control flow:**
- _[Describe how agents interact: sequential? loop with evaluator? parallel drafting?]_

**Termination conditions:**
- _[When does the system stop? Quality threshold? Max iterations?]_

**Estimated LLM calls:** _[Range: best case to worst case]_

#### RAG Integration (Optional)

If you plan to incorporate **RAG (Retrieval-Augmented Generation)** to ground your proposals in existing knowledge:

**What knowledge sources would you use?**
- _[e.g., Previous successful proposals, domain best practices, Azure architecture docs, company templates]_

**Where in your workflow/agents would RAG be used?**
- _[e.g., Planner retrieves similar proposals; Drafter retrieves domain-specific terminology]_

**Retrieval strategy:**
- _[Semantic search? Keyword matching? Hybrid? When to retrieve?]_

---

### 5.3 Evaluation Plan

How will you determine if a generated proposal is "good enough"? Define your quality criteria and evaluation approach.

#### Evaluation Dimensions

Rate the importance of each dimension for your use case (Low / Medium / High):

| Dimension | Importance | How You'll Measure |
|-----------|------------|--------------------|
| **Relevance to requirement** | | _[Does it address all stated needs?]_ |
| **Clarity and structure** | | _[Is it well-organized and easy to read?]_ |
| **Technical correctness** | | _[Are Azure services and approaches feasible?]_ |
| **Completeness** | | _[Are all expected sections present?]_ |
| **Professionalism / tone** | | _[Appropriate business language? No errors?]_ |
| **Specificity** | | _[Concrete details vs. vague generalities?]_ |
| **Feasibility** | | _[Can this actually be implemented?]_ |

#### Evaluation Mechanism

Choose your approach and document the details:

**Primary evaluation method:** _[Manual review / LLM-as-judge / Hybrid / Automated checklist]_

**If using LLM-as-judge:**
- Will the same model evaluate, or a different one?
- What scoring rubric will you use? (e.g., 1-10 scale, pass/fail per dimension)
- Who reviews the LLM's evaluation? (Human-in-the-loop?)

**If using manual review:**
- Who reviews? (Subject matter experts? Proposal team?)
- What checklist or criteria do they use?

**Scoring thresholds or rules:**
- _[e.g., "Minimum score of 7/10 on all dimensions" or "Pass all required criteria"]_

**Feedback loop:**
- If a proposal doesn't meet quality standards, what happens?
  - _[Generate a new version? Refine specific sections? Human revision?]_

**Example evaluation criteria:**
```markdown
# Proposal Quality Rubric

1. Relevance (0-10): Does it address the client's stated problem?
2. Clarity (0-10): Is it well-structured and understandable?
3. Feasibility (0-10): Is the solution technically sound and achievable?
4. Completeness (0-10): Are all required sections present with sufficient detail?

Passing score: Average ≥ 7.5, no dimension below 6
```

---

---

## 💻 Implementation Scratch Area (Optional Coding)

This section is **completely optional**. If you want hands-on practice implementing your design before seeing the reference solution, use these cells as a starting point.

### 6.1 Option A – Workflow Implementation (Optional)

Use this area if you chose a **workflow/chain** approach with fixed, sequential steps.

In [ ]:
# TODO: (Optional) Implement step 1 - Generate outline
# Hint: Prompt the LLM to create a structured outline of the proposal sections

def generate_outline(requirement_text: str) -> str:
    """
    Generate a structured outline for the proposal based on requirements.
    
    Args:
        requirement_text: The client requirement description
    
    Returns:
        A structured outline with sections and key points
    """
    
    # Your implementation here:
    prompt = f"Based on this requirement, create a proposal outline:\n\n{requirement_text}\n\nOutline:"
    return call_llm(prompt, temperature=0.3)

In [ ]:
# TODO: (Optional) Implement step 2 - Generate individual sections
# Hint: For each section, prompt the LLM with requirement + outline context

def generate_section(requirement_text: str, outline: str, section_name: str) -> str:
    """
    Generate content for a specific proposal section.
    
    Args:
        requirement_text: The client requirement
        outline: The proposal outline from step 1
        section_name: Which section to generate (e.g., "Executive Summary")
    
    Returns:
        Generated content for that section
    """
    
    # Your implementation here:
    prompt = f"Requirement: {requirement_text}\n\nOutline: {outline}\n\nWrite the {section_name} section:"
    return call_llm(prompt, temperature=0.7, max_tokens=800)

In [ ]:
# TODO: (Optional) Implement workflow orchestrator
# Hint: Call generate_outline, then generate_section for each section, then combine

def build_proposal_workflow(requirement_text: str) -> str:
    """
    End-to-end workflow to generate a complete proposal.
    
    Args:
        requirement_text: The client requirement
    
    Returns:
        Complete proposal document
    """
    # Your implementation here:
    # Step 1: Generate outline
    outline = generate_outline(requirement_text)
    #
    # Step 2: Generate each section
    sections = {}
    for section_name in ["Executive Summary", "Problem Statement", "Proposed Solution", ...]:
        sections[section_name] = generate_section(requirement_text, outline, section_name)
    
    # Step 3: Combine into final document
    proposal = "\n\n".join([f"## {name}\n{content}" for name, content in sections.items()])
    return proposal


# Test your workflow (uncomment to run)
sample_req = """Document Intelligence Assistant Client / Business Unit: Legal Operations Department, ABC Financial Services
Domain / Industry: Financial Services / Banking
Problem Statement:Our legal team processes hundreds of contracts, compliance documents, and regulatory filings each month. Currently, lawyers spend 40-60% of their time on manual document review: extracting key clauses, identifying risks, checking compliance, and summarizing findings for stakeholders. This is unsustainable as document volume grows 20% year-over-year.
Desired Outcomes:- Reduce document review time by 50-70%
- Improve consistency in risk identification across reviewers
- Enable faster response to regulatory queries
- Free up lawyers to focus on high-value strategic work
- Maintain or improve accuracy (currently 95% accuracy on manual review)
Constraints:
- Budget: $200K-$300K for MVP (6-month project)
- Technology: Must use Azure (existing enterprise agreement), prefer Azure OpenAI Service
- Compliance: Subject to financial services regulations (SOC 2, data residency requirements)
- Timeline: Need MVP in production within 6 months
Non-Functional Requirements:
- Performance: Process a 50-page contract in under 2 minutes
- Security: End-to-end encryption, role-based access control, audit logging
- Scalability: Handle 500 documents per day initially, scale to 2000/day within 12 months
- Integration: Must integrate with existing document management system (SharePoint) and case management tool
"""
result = build_proposal_workflow(sample_req)
print(result)

---

### 6.2 Option B – Agentic Implementation (Optional)

Use this area if you chose an **agent** or **agentic** approach with dynamic decision-making and roles.

In [ ]:
# Hint: Analyzes requirement and creates a structured plan

def planner_agent(requirement_text: str) -> dict:
    """
    Planner agent: Analyzes requirement and produces structured plan
    """
    print("\n[Planner] Creating plan...")

    prompt = f"""
Analyze this requirement and create a structured proposal plan.

Requirement:
{requirement_text}

Return clearly:
1. Key priorities
2. Recommended proposal sections
3. Risks to address
4. Technical approach focus
"""

    plan_text = call_llm(prompt, temperature=0.3)

    print("[Planner] Plan created ✔")
    return {"plan": plan_text}

In [ ]:
# Hint: Generates proposal content based on requirement and plan
def drafter_agent(requirement_text: str, plan: dict) -> str:
    """
    Generates proposal draft from requirement + plan
    """
    print("\n[Drafter] Writing proposal draft...")

    prompt = f"""
Write a professional proposal.

Requirement:
{requirement_text}

Plan:
{plan["plan"]}

Include sections:
- Executive Summary
- Proposed Solution
- Architecture
- Timeline
- Risks
- Cost Estimate
"""

    draft = call_llm(prompt, temperature=0.7, max_tokens=2000)

    print("[Drafter] Draft generated ✔")
    return draft

In [ ]:
# Hint: Scores the draft and provides feedback for improvement
def evaluator_agent(requirement_text: str, draft_proposal: str) -> dict:
    """
    Evaluates draft quality
    """
    print("\n[Evaluator] Evaluating proposal...")

    prompt = f"""
Evaluate this proposal against requirement.

Requirement:
{requirement_text}

Proposal:
{draft_proposal}

Score 0-10:
Relevance:
Clarity:
Completeness:
Feasibility:

Return FINAL_SCORE as average.
"""

    evaluation = call_llm(prompt, temperature=0.2)

    # simple pass logic for demo
    passed = "FINAL_SCORE: 8" in evaluation or "FINAL_SCORE: 9" in evaluation or "FINAL_SCORE: 10" in evaluation

    print("[Evaluator] Evaluation complete ✔")
    return {
        "evaluation": evaluation,
        "pass": passed
    }

In [ ]:
sample_req = """Document Intelligence Assistant Client / Business Unit: Legal Operations Department, ABC Financial Services
Domain / Industry: Financial Services / Banking
Problem Statement:Our legal team processes hundreds of contracts, compliance documents, and regulatory filings each month. Currently, lawyers spend 40-60% of their time on manual document review: extracting key clauses, identifying risks, checking compliance, and summarizing findings for stakeholders. This is unsustainable as document volume grows 20% year-over-year.
Desired Outcomes:- Reduce document review time by 50-70%
- Improve consistency in risk identification across reviewers
- Enable faster response to regulatory queries
- Free up lawyers to focus on high-value strategic work
- Maintain or improve accuracy (currently 95% accuracy on manual review)
Constraints:
- Budget: $200K-$300K for MVP (6-month project)
- Technology: Must use Azure (existing enterprise agreement), prefer Azure OpenAI Service
- Compliance: Subject to financial services regulations (SOC 2, data residency requirements)
- Timeline: Need MVP in production within 6 months
Non-Functional Requirements:
- Performance: Process a 50-page contract in under 2 minutes
- Security: End-to-end encryption, role-based access control, audit logging
- Scalability: Handle 500 documents per day initially, scale to 2000/day within 12 months
- Integration: Must integrate with existing document management system (SharePoint) and case management tool
"""

In [ ]:
# Hint: Orchestrates planner → drafter → evaluator, with potential iteration

def run_agentic_pipeline(requirement_text: str, max_iterations: int = 3) -> str:
    """
    Runs planner → drafter → evaluator loop
    """

    print("\n" + "="*60)
    print("AGENTIC PIPELINE STARTED")
    print("="*60)

    # STEP 1 — Plan
    plan = planner_agent(requirement_text)

    # STEP 2 — Draft Loop
    for i in range(max_iterations):
        print(f"\n--- ITERATION {i+1} ---")

        draft = drafter_agent(requirement_text, plan)
        evaluation = evaluator_agent(requirement_text, draft)

        print("\nEvaluation Result:")
        print(evaluation["evaluation"])

        if evaluation["pass"]:
            print("\n✅ Quality threshold met. Stopping loop.")
            return draft

        print("\n⚠️ Draft needs improvement. Retrying...")

    print("\n⚠️ Max iterations reached. Returning best draft.")
    return draft

In [ ]:
result = run_agentic_pipeline(sample_req)

print("\n" + "="*60)
print("FINAL OUTPUT")
print("="*60)
print(result)

---

## 🔍 Prepare for the Walkthrough

Congratulations on completing your design! Whether you sketched out ideas on paper, filled in the templates, or wrote some code, you now have a structured approach to the Proposal Assistant use case.

### What's Next?

In that notebook, you will find:

1. **Workflow-Style Implementation**
   - Complete implementation using only the OpenAI SDK (no LangChain)
   - Fixed, predictable pipeline: outline → draft sections → combine → format
   - Clear cost and token analysis
   - Approximately 7-9 LLM calls per proposal

2. **Agentic-Style Implementation**
   - Complete implementation using only the OpenAI SDK (no LangChain)
   - Dynamic approach with planner, drafter, and evaluator agents
   - Iterative refinement loop based on quality scores
   - Approximately 5-12 LLM calls (varies based on quality threshold)

3. **Comparison & Analysis**
   - Side-by-side comparison of both approaches
   - Discussion of trade-offs (cost vs. quality, predictability vs. flexibility)
   - Guidance on when to use each approach

### How to Compare

As you study the reference implementations, ask yourself:

- **Similarities:** What parts of your design match the reference solution?
- **Differences:** Where did you make different choices? Why might the reference solution have taken a different approach?
- **Surprises:** Did anything unexpected work particularly well or poorly?
- **Applicability:** Which approach (workflow vs. agentic) feels more suitable for your real-world scenarios?

---

## 💭 Reflection

Take a moment to capture your thoughts and questions before moving on to the reference solution.

### Reflection Prompts

**1. Architecture Choice**

Did your initial choice (prompt app / workflow / agent / hybrid) still feel right after designing the details? Why or why not?

_Your reflection here..._

---

**2. Deterministic vs. Adaptive**

Which part of this use case do you feel benefits most from a **deterministic workflow** (fixed steps), and which part (if any) might benefit from an **agentic approach** (dynamic decisions)?

_Your reflection here..._

---

**3. RAG Integration**

If you were to extend this solution with **RAG** (retrieving previous proposals, domain guidelines, architecture patterns, etc.), where would you plug it into your design? At what stages would retrieval be most valuable?

_Your reflection here..._

---

**4. Quality vs. Cost**

How would you balance **proposal quality** against **cost and latency**? Would you prefer a fast, cheaper workflow or a slower, more expensive agentic system that iteratively refines quality?

_Your reflection here..._

---

**5. Curiosity & Questions**

What are you most curious to see in the reference solution? What questions do you hope it will answer?

_Your questions here..._

---

### Your Notes (Free-Form)

Use this space for any additional thoughts, diagrams, pseudocode, or observations:

_Your notes here..._

---

## 🎓 Summary & Next Steps

You've completed the workspace notebook! Here's what you accomplished:

✅ Understood the Proposal Assistant use case and requirements  
✅ Evaluated architecture patterns and chose an approach  
✅ Designed inputs, outputs, steps/roles, and evaluation criteria  
✅ (Optional) Started implementing your solution  
✅ Reflected on your design choices

**Happy learning! 🚀**